# Atividade: prevendo a progressão do diabetes

Nas duas últimas semanas, o modelo respondia com o nome de uma fruta. Agora ele vai responder com um número: quanto o diabetes de um paciente progrediu um ano depois, a partir de dez medidas tomadas no início. Esse é um problema de **regressão**.

Você vai implementar a regressão linear do zero, de duas maneiras: pela fórmula fechada dos mínimos quadrados e pelo gradiente descendente. Depois vai escolher atributos com validação cruzada, olhar os resíduos e testar o modelo em pacientes novos.

`pandas` e `numpy` estão liberados para organizar os dados e fazer as contas — inclusive `np.linalg.solve` para resolver um sistema linear —, e `matplotlib` para os gráficos. O ajuste, o gradiente, a validação e as métricas são seus: não use bibliotecas que já forneçam o algoritmo.

Execute as células na ordem. Onde houver `...`, o código é seu; os comentários em volta indicam o que cada trecho precisa fazer.

## 1. Conhecer os dados

O conjunto **Diabetes** é distribuído com o scikit-learn. Usamos os valores originais, sem padronização. Cada linha é um paciente:

| Coluna | Significado |
| :-- | :-- |
| `age` | idade, em anos |
| `sex` | sexo, codificado como 1 ou 2 |
| `bmi` | índice de massa corporal (IMC) |
| `bp` | pressão arterial média |
| `s1` a `s6` | exames de sangue: colesterol total, LDL, HDL, razão colesterol/HDL, triglicerídeos (log) e glicemia |
| `progression` | medida da progressão da doença um ano depois — é o que queremos prever |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TRAIN_URL = 'https://raw.githubusercontent.com/H2IA/treinamento/main/data/diabetes/train.csv'
train = pd.read_csv(TRAIN_URL)

FEATURES = ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
TARGET = 'progression'

y_train = train[TARGET].to_numpy(dtype=float)

print(f'{len(train)} pacientes')
train.describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7), sharey=True)
for ax, feature in zip(axes.flat, FEATURES):
    ax.scatter(train[feature], y_train, s=10, alpha=0.5)
    ax.set_xlabel(feature)
for ax in axes[:, 0]:
    ax.set_ylabel(TARGET)
plt.tight_layout()
plt.show()

**Antes de programar, responda:** se você não soubesse nada sobre um paciente, qual seria o melhor palpite para a progressão? Qual das medidas parece mais útil nos gráficos? Alguma parece não ajudar em nada?

**Minha resposta:** ...

## 2. Uma reta, pela fórmula

Com um único atributo, o modelo é `ŷ = w · x + b`. A reta que minimiza o erro quadrático médio tem solução fechada:

$$w = \frac{\sum_i (x_i - \bar{x})(y_i - \bar{y})}{\sum_i (x_i - \bar{x})^2} \qquad b = \bar{y} - w\,\bar{x}$$

In [ ]:
def fit_simple(x, y):
    """Recebe dois arrays do mesmo tamanho e devolve (w, b) por mínimos quadrados."""
    # Calcule as médias de x e de y, depois a inclinação pela fórmula acima
    # e, por fim, o viés.
    ...

In [ ]:
# Confira com o exemplo do roteiro antes de seguir.
print(fit_simple(np.array([22, 26, 30]), np.array([110, 160, 180])))  # esperado: (8.75, -77.5)

In [ ]:
w_bmi, b_bmi = fit_simple(train['bmi'], y_train)
print(f'progressão ≈ {w_bmi:.2f} · IMC + {b_bmi:.2f}')  # esperado: w ≈ 10.13, b ≈ -114.54

xs = np.linspace(train['bmi'].min(), train['bmi'].max(), 100)
plt.scatter(train['bmi'], y_train, s=10, alpha=0.5)
plt.plot(xs, w_bmi * xs + b_bmi, color='black')
plt.xlabel('bmi')
plt.ylabel(TARGET)
plt.show()

**Responda:**

1. Quanto a progressão prevista aumenta para cada ponto a mais de IMC?
2. O viés `b` tem algum significado? O que o modelo prevê para um IMC de zero?
3. Os pontos ficam perto da reta?

**Minhas respostas:** ...

## 3. Métricas

- MAE, o erro absoluto médio: `mean(|y − ŷ|)`;
- RMSE, a raiz do erro quadrático médio: `sqrt(mean((y − ŷ)²))`;
- R², a comparação com o palpite da média: `1 − Σ(y − ŷ)² / Σ(y − ȳ)²`.

MAE e RMSE estão na unidade da progressão. O R² não tem unidade.

In [ ]:
def mae(y_true, y_pred):
    ...


def rmse(y_true, y_pred):
    ...


def r2(y_true, y_pred):
    ...


def report(y_true, y_pred):
    return {'MAE': mae(y_true, y_pred), 'RMSE': rmse(y_true, y_pred), 'R2': r2(y_true, y_pred)}

In [ ]:
results = {'média do treino': report(y_train, np.full(len(y_train), y_train.mean()))}

for feature in ['bmi', 's5', 'bp']:
    w, b = fit_simple(train[feature], y_train)
    results[f'só {feature}'] = report(y_train, w * train[feature] + b)

pd.DataFrame(results).T.round(3)

**Responda:**

1. Por que o R² da média do treino é exatamente zero?
2. Qual medida isolada explica melhor a progressão? Quanto da variação ela deixa sem explicar?
3. Compare o MAE e o RMSE de um mesmo modelo. Por que o RMSE é sempre maior ou igual?

**Minhas respostas:** ...

## 4. A mesma reta, pelo gradiente

Com vários atributos, o modelo é `ŷ = X · w + b`, onde `X` tem uma coluna por atributo e `w` tem um peso por coluna. Derive no papel as derivadas do MSE antes de olhar a resposta abaixo, e confira com o vídeo do StatQuest.

<details>
<summary>Conferir as derivadas</summary>

$$\frac{\partial\,\text{MSE}}{\partial w_j} = \frac{2}{n} \sum_i (\hat{y}_i - y_i)\, x_{ij} \qquad \frac{\partial\,\text{MSE}}{\partial b} = \frac{2}{n} \sum_i (\hat{y}_i - y_i)$$

</details>

In [ ]:
def gradient_descent(X, y, lr, n_steps):
    """X é um array (n, d), y um array (n,).
    Devolve (w, b, history), onde history é a lista do MSE a cada passo."""
    n, d = X.shape
    w = np.zeros(d)
    b = 0.0
    history = []

    for step in range(n_steps):
        # 1. calcule as previsões com os pesos atuais
        # 2. guarde o MSE em history
        # 3. calcule as duas derivadas
        # 4. atualize w e b dando um passo de tamanho lr na direção oposta
        ...

    return w, b, history

Rode primeiro com o IMC sozinho, **sem padronizar**, e compare com a fórmula. Com os valores brutos, a taxa de aprendizado precisa ser pequena e o número de passos, grande. Comece com os valores abaixo e experimente: quantos passos são necessários para os pesos coincidirem? O que acontece com `lr=0.002`?

In [ ]:
X_bmi = train[['bmi']].to_numpy(dtype=float)

w, b, history = gradient_descent(X_bmi, y_train, lr=0.001, n_steps=200_000)
print(f'gradiente: w = {w[0]:.2f}, b = {b:.2f}')
print(f'fórmula:   w = {w_bmi:.2f}, b = {b_bmi:.2f}')

### Padronizar

Padronizar um atributo é subtrair sua média e dividir pelo seu desvio padrão. Depois disso, todos os atributos ficam com média zero e a mesma escala.

Calcule a média e o desvio padrão **só no treino** e guarde-os: os pacientes de teste serão transformados com os mesmos números.

In [ ]:
def fit_scaler(X):
    """Devolve (mean, std) de cada coluna de X."""
    ...


def transform(X, mean, std):
    ...

In [ ]:
X_raw = train[FEATURES].to_numpy(dtype=float)
mean, std = fit_scaler(X_raw)
X_std = transform(X_raw, mean, std)

# Experimente várias taxas nos dois casos. Encontre uma que diverge e uma que
# converge devagar demais. Se a curva sumir do gráfico, o erro explodiu.
experiments = {
    'bruto, lr=1e-5': (X_raw, 1e-5),
    'bruto, lr=1e-4': (X_raw, 1e-4),
    'padronizado, lr=0.01': (X_std, 0.01),
    'padronizado, lr=0.1': (X_std, 0.1),
}

for label, (X, lr) in experiments.items():
    with np.errstate(over='ignore', invalid='ignore'):
        _, _, history = gradient_descent(X, y_train, lr=lr, n_steps=1000)
    plt.plot(history, label=label)

plt.yscale('log')
plt.ylim(1e3, 1e5)
plt.xlabel('passo')
plt.ylabel('MSE no treino')
plt.legend()
plt.show()

**Responda:**

1. Quantos passos o gradiente precisou com o IMC bruto? E com os dez atributos padronizados?
2. Qual taxa fez o erro explodir? Por que uma taxa grande demais diverge, em vez de só convergir mais rápido?

**Minhas respostas:** ...

## 5. Vários atributos, pela fórmula

Para vários atributos, a solução exata de mínimos quadrados vem das **equações normais**:

$$(X^\top X)\, w = X^\top y$$

Para incluir o viés, acrescente a `X` uma primeira coluna de uns: o peso dessa coluna será `b`. Use os atributos padronizados, para que os pesos possam ser comparados com os do gradiente.

In [ ]:
def fit_normal(X, y):
    """Devolve (w, b) resolvendo as equações normais."""
    # 1. monte A = [1 | X], com uma coluna de uns na frente
    # 2. resolva (A^T A) theta = A^T y com np.linalg.solve
    # 3. o primeiro elemento de theta é b; os outros são w
    ...

In [ ]:
w_exact, b_exact = fit_normal(X_std, y_train)

comparison = {'fórmula': np.append(w_exact, b_exact)}
mse = {'fórmula': np.mean((X_std @ w_exact + b_exact - y_train) ** 2)}

# Aumente o número de passos e observe os pesos de s1 e s2.
for n_steps in [100, 1000, 10_000]:
    w, b, history = gradient_descent(X_std, y_train, lr=0.1, n_steps=n_steps)
    comparison[f'gradiente, {n_steps} passos'] = np.append(w, b)
    mse[f'gradiente, {n_steps} passos'] = history[-1]

display(pd.DataFrame(comparison, index=FEATURES + ['b']).round(2))
pd.Series(mse, name='MSE no treino').round(2)

In [ ]:
# O peso de s1 sozinho e acompanhado. A correlação ajuda a explicar a diferença.
w_s1, _ = fit_simple(train['s1'], y_train)
print(f'peso de s1 sozinho (valores brutos): {w_s1:.3f}')
print(f'peso de s1 com os dez atributos (valores brutos): {w_exact[FEATURES.index("s1")] / std[FEATURES.index("s1")]:.3f}')
train[['s1', 's2', 's3', 's4']].corr().round(2)

**Responda:**

1. O gradiente e a fórmula chegaram ao mesmo MSE? E aos mesmos pesos? Quais pesos demoram mais para convergir?
2. Enquanto os pesos de `s1` e `s2` ainda mudavam, quanto mudava o MSE? O que isso diz sobre a forma da "tigela" nessa direção?
3. O sinal do peso de `s1` mudou quando os outros atributos entraram? O que esse peso negativo quer dizer — e o que ele não quer dizer?

**Minhas respostas:** ...

## 6. Escolher os atributos

Para comparar modelos, use a validação cruzada com `k = 5` partes. Sorteie a ordem dos pacientes uma vez, divida-os em cinco grupos e, para cada grupo, ajuste o modelo com os outros quatro e preveja o que ficou de fora. Assim cada paciente recebe exatamente uma previsão feita por um modelo que não o viu.

Junte padronização e gradiente em um par de funções `fit` e `predict`. Cuidado: dentro da validação, a média e o desvio padrão precisam ser calculados apenas com os pacientes de treino daquela rodada.

In [ ]:
def fit(X, y, lr=0.1, n_steps=1000):
    """Padroniza X, ajusta por gradiente descendente e devolve um modelo."""
    # Um dicionário com mean, std, w e b é suficiente.
    ...


def predict(model, X):
    ...


def cv_predictions(X, y, k=5, seed=0, **fit_args):
    """Devolve, para cada linha, a previsão de um modelo ajustado sem o grupo dela.
    fit_args é repassado para fit (por exemplo, lr e n_steps)."""
    order = np.random.default_rng(seed).permutation(len(y))
    folds = np.array_split(order, k)
    preds = np.zeros(len(y))
    for fold in folds:
        # monte uma máscara que exclui as linhas de fold, ajuste com as outras
        # e preveja as linhas de fold
        ...
    return preds

### Construir um atributo

Proponha ao menos um atributo novo — um quadrado ou um produto entre dois atributos — que você espera que ajude. Escreva a justificativa antes de testar.

**Meu atributo e por que ele deveria ajudar:** ...

In [ ]:
def add_features(df):
    """Recebe um DataFrame com as medidas e devolve uma cópia com os atributos novos.
    Será aplicada também ao teste, então use apenas as colunas de medidas."""
    df = df.copy()
    # df['meu_atributo'] = ...
    ...
    return df


train = add_features(train)
train.head()

In [ ]:
# Acrescente aqui os modelos que quiser comparar, incluindo os seus.
candidates = {
    'bmi': ['bmi'],
    'bmi + s5': ['bmi', 's5'],
    'bmi + bp + s5': ['bmi', 'bp', 's5'],
    'todos': FEATURES,
    # 'todos + meu atributo': FEATURES + ['meu_atributo'],
}

cv = {}
rows = {}
for label, cols in candidates.items():
    X = train[cols].to_numpy(dtype=float)
    cv[label] = cv_predictions(X, y_train)
    rows[label] = {
        'RMSE treino': rmse(y_train, predict(fit(X, y_train), X)),
        'RMSE validação': rmse(y_train, cv[label]),
        'R2 validação': r2(y_train, cv[label]),
    }

pd.DataFrame(rows).T.round(3)

**Responda:**

1. Qual modelo teve o menor erro na validação? A diferença para o segundo colocado é grande?
2. A distância entre o erro no treino e o da validação cresce conforme o modelo ganha atributos?
3. Seu atributo construído ajudou?
4. Qual modelo você escolheria, e por quê?

**Minhas respostas:** ...

## 7. Olhar os resíduos

Escolha o modelo que você levaria para o teste e analise os resíduos da validação, `y − ŷ`. Um resíduo positivo significa que o modelo **subestimou** a progressão.

In [ ]:
CHOSEN = 'todos'  # troque pelo nome do modelo escolhido em candidates

residuals = y_train - cv[CHOSEN]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].scatter(cv[CHOSEN], residuals, s=10, alpha=0.5)
axes[0].set_xlabel('progressão prevista')
axes[1].scatter(train['bmi'], residuals, s=10, alpha=0.5)
axes[1].set_xlabel('bmi')
axes[0].set_ylabel('resíduo (real − previsto)')
for ax in axes:
    ax.axhline(0, color='black', linewidth=1)
plt.show()

train.assign(res=residuals).groupby('sex')['res'].agg(['mean', 'std', 'count']).round(1)

**Responda:**

1. O tamanho dos erros é parecido em toda a faixa de previsões, ou muda? Onde o modelo erra mais?
2. Os resíduos contra o IMC formam alguma curva?
3. Faz sentido usar o código 1 ou 2 de `sex` como se fosse uma medida? O que mudaria se houvesse três categorias?

**Minhas respostas:** ...

## 8. Congelar o modelo

Antes de abrir os pacientes novos:

1. registre o modelo escolhido e o RMSE dele na validação;
2. estime o RMSE que ele terá nos 88 pacientes de teste;
3. salve esta versão do notebook e não altere mais o modelo até avaliar o teste.

**Meu modelo:** ...

**Minha estimativa de RMSE no teste:** ...

**Minha justificativa:** ...

## 9. Prever os pacientes novos

O modelo final é ajustado com os 354 pacientes. O arquivo de teste não traz a progressão.

In [ ]:
TEST_URL = 'https://raw.githubusercontent.com/H2IA/treinamento/main/data/diabetes/test.csv'
test = add_features(pd.read_csv(TEST_URL))

final_cols = candidates[CHOSEN]
final_model = fit(train[final_cols].to_numpy(dtype=float), y_train)

test_predictions = test[['id']].copy()
test_predictions['prediction'] = predict(final_model, test[final_cols].to_numpy(dtype=float))
test_predictions.to_csv('progression_predictions.csv', index=False)
display(test_predictions.round(1).head())

try:
    from google.colab import files
    files.download('progression_predictions.csv')
except ImportError:
    print('The file progression_predictions.csv was saved in the current folder.')

## 10. Avaliar depois da liberação do gabarito

Cole abaixo o link direto fornecido pelo professor. Execute esta seção somente depois de entregar ou congelar suas previsões.

In [ ]:
ANSWER_KEY_URL = ''  # Cole aqui o link direto liberado pelo professor.

if not ANSWER_KEY_URL.strip():
    print('O gabarito ainda não foi informado. Preserve suas previsões e volte depois.')
else:
    answer_key = pd.read_csv(ANSWER_KEY_URL)
    test_results = test_predictions.merge(answer_key, on='id', validate='one_to_one')
    y_test = test_results[TARGET].to_numpy(dtype=float)

    print('Modelo escolhido no teste:')
    display(pd.Series(report(y_test, test_results['prediction'])).round(3))

    # Compare o modelo escolhido com os outros candidatos.
    # Todos são ajustados com os 354 pacientes, como o modelo final.
    test_rows = {}
    for label, cols in candidates.items():
        model = fit(train[cols].to_numpy(dtype=float), y_train)
        test_rows[label] = {
            'RMSE validação': rows[label]['RMSE validação'],
            'RMSE teste': rmse(y_test, predict(model, test[cols].to_numpy(dtype=float))),
        }
    display(pd.DataFrame(test_rows).T.round(1))

## 11. Muitos atributos (recomendada)

Esta parte é opcional, mas é aqui que aparece, em regressão, o mesmo fenômeno da árvore sem limite.

Construa todos os quadrados e produtos entre pares dos dez atributos, além dos próprios atributos. Descarte `sex²`: com só dois valores, ele repete a coluna `sex`.

In [ ]:
from itertools import combinations_with_replacement


def all_products(df):
    out = df[FEATURES].copy()
    for a, b in combinations_with_replacement(FEATURES, 2):
        if a == b == 'sex':
            continue
        out[f'{a}*{b}'] = df[a] * df[b]
    return out


X_many = all_products(train).to_numpy(dtype=float)
print(X_many.shape)

In [ ]:
def fit_exact(X, y):
    """Como fit, mas usando as equações normais no lugar do gradiente."""
    mean, std = fit_scaler(X)
    w, b = fit_normal(transform(X, mean, std), y)
    return {'mean': mean, 'std': std, 'w': w, 'b': b}


def cv_exact(X, y, k=5, seed=0):
    """Como cv_predictions, mas ajustando com fit_exact."""
    ...

In [ ]:
X_ten = train[FEATURES].to_numpy(dtype=float)

many = {
    'dez atributos, fórmula': (
        r2(y_train, predict(fit_exact(X_ten, y_train), X_ten)),
        r2(y_train, cv_exact(X_ten, y_train)),
    ),
    f'{X_many.shape[1]} atributos, fórmula': (
        r2(y_train, predict(fit_exact(X_many, y_train), X_many)),
        r2(y_train, cv_exact(X_many, y_train)),
    ),
}

# Agora o gradiente, com poucos passos. Com lr=0.1 ele diverge: descubra por quê
# e escolha uma taxa que funcione.
LR_MANY = 0.1
with np.errstate(over='ignore', invalid='ignore'):
    many[f'{X_many.shape[1]} atributos, gradiente'] = (
        r2(y_train, predict(fit(X_many, y_train, lr=LR_MANY), X_many)),
        r2(y_train, cv_predictions(X_many, y_train, lr=LR_MANY)),
    )

pd.DataFrame(many, index=['R2 treino', 'R2 validação']).T.round(3)

**Responda:**

1. O que aconteceu com o R² no treino e na validação quando os atributos passaram de 10 para 64?
2. Por que a taxa que funcionava com dez atributos diverge com 64?
3. O gradiente interrompido cedo validou melhor que a solução exata? Explique, pensando no que os pesos ainda não tiveram tempo de fazer.

**Minhas respostas:** ...

## 12. Fechamento

Responda com suas palavras:

1. A fórmula e o gradiente descendente chegaram aos mesmos pesos? O que a padronização mudou no gradiente descendente — e o que ela não mudou nas previsões finais?
2. O que o peso do IMC no modelo de um atributo significa? Ele continua igual quando os outros atributos entram? Por quê?
3. O que aconteceu com o peso de `s1` quando `s2` entrou no modelo? Você usaria os pesos desse modelo para dizer qual exame "importa mais"?
4. O R² do seu melhor modelo ficou longe de 1. Isso indica um erro na implementação, um modelo ruim ou um limite dos dados? Como você distinguiria esses casos?
5. Que padrão você encontrou nos resíduos? O que ele sugere que o modelo não captura?
6. O resultado no teste ficou acima ou abaixo da sua estimativa? A ordem dos modelos se manteve?
7. Um paciente novo tem IMC 48, acima de qualquer paciente do treino. Você confiaria na previsão do seu modelo para ele? O que pode dar errado quando um modelo é usado fora da faixa dos exemplos em que foi ajustado?
8. Se este modelo fosse usado para decidir quais pacientes recebem acompanhamento mais próximo, que tipo de erro seria mais grave: superestimar ou subestimar a progressão? Alguma das métricas que você implementou distingue os dois?